In [ ]:
from transformers import ViTFeatureExtractor, ViTForImageClassification, TrainingArguments, Trainer, AdamW
from sklearn.model_selection import train_test_split
from datasets import Dataset
from PIL import Image
import os
import torch
import random
import kagglehub

In [ ]:
# Constants
num_classes = 30  # Number of fish species
image_size = 224  # Input size for Vision Transformer
batch_size = 8  # Reduced batch size to avoid memory issues
epochs = 5
learning_rate = 5e-5

# Dataset paths
dataset_path = kagglehub.dataset_download("jorritvenema/affine")
dataset_path = os.path.join(dataset_path, "dataset")
print(f"Dataset path: {dataset_path}")

# Load images and labels
def load_images_and_labels(dataset_path):
    image_paths, labels = [], []
    species_folders = [os.path.join(dataset_path, folder) for folder in os.listdir(dataset_path)
                       if os.path.isdir(os.path.join(dataset_path, folder))]
    for idx, species_folder in enumerate(species_folders):
        images = [os.path.join(species_folder, file) for file in os.listdir(species_folder)
                  if file.endswith(('.jpg', '.jpeg', '.png'))]
        image_paths.extend(images)
        labels.extend([idx] * len(images))
    return image_paths, labels

image_paths, labels = load_images_and_labels(dataset_path)

# Split dataset into train, validation, and test sets
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.2, stratify=train_labels, random_state=42
)

# Initialize feature extractor
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')

# Preprocessing function for Hugging Face Dataset
def preprocess_function(examples):
    images = []
    for img_path in examples['image_path']:
        try:
            img = Image.open(img_path).convert("RGB")
            images.append(img)
        except Exception as e:
            print(f"Error processing image {img_path}: {e}")
            images.append(None)
    processed = feature_extractor(images=images, return_tensors="pt")
    return {"pixel_values": processed["pixel_values"], "labels": examples["labels"]}

# Create Hugging Face Dataset
def create_hf_dataset(image_paths, labels):
    dataset_dict = {"image_path": image_paths, "labels": labels}
    dataset = Dataset.from_dict(dataset_dict)
    return dataset.map(preprocess_function, batched=True, batch_size=batch_size)


In [ ]:
# Process datasets
print("Processing training data...")
train_dataset = create_hf_dataset(train_paths, train_labels)
print("Processing validation data...")
val_dataset = create_hf_dataset(val_paths, val_labels)
print("Processing test data...")
test_dataset = create_hf_dataset(test_paths, test_labels)

In [ ]:
# Debugging: Use a subset of the dataset during testing
# train_dataset = train_dataset.select(range(1000))  # Use 1000 samples for training
# val_dataset = val_dataset.select(range(200))  # Use 200 samples for validation

# Load the pre-trained Vision Transformer model
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=num_classes
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Freeze the backbone (optional)
# for param in model.vit.parameters():
#     param.requires_grad = False  # Uncomment to freeze backbone layers

# Define optimizer with differential learning rates
optimizer = AdamW([
    {"params": model.vit.parameters(), "lr": 1e-5},  # Slower learning rate for backbone
    {"params": model.classifier.parameters(), "lr": 5e-5}  # Faster learning rate for classifier
])

# Define training arguments with optimizations
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=2,  # Simulate a larger batch size (batch size 16)
    num_train_epochs=epochs,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
    fp16=True,  # Enable mixed precision training
    dataloader_num_workers=2,  # Use 4 workers for data loading
    report_to="none"
)

# Define metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits).to(device), dim=-1).cpu().numpy()
    labels = torch.tensor(labels).cpu().numpy()
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)  # Use the custom optimizer
)




In [ ]:
# Train the model
print("Starting training...")
trainer.train()

# Evaluate on the test set
print("Evaluating on test set...")
metrics = trainer.evaluate(test_dataset)
print(f"Test set metrics: {metrics}")